# `do_eyetracking.ipynb`

**Description:** Temporal alignment of Pupil Labs gaze data to video frames, producing the `aligned_gaze.csv` files used as input to the computer vision pipeline (`run_panoptic_tracking.py`).  
**Author:** Raquel Moleiro Marques (s243636)  
**Date:** June 2026

### Imports

In [2]:
import os
import cv2
import torch
import numpy as np
from utils import *
import pandas as pd
import seaborn as sns
from modules import *
from PIL import Image
import matplotlib.pyplot as plt
from pycocotools.coco import COCO
import utils.for_eye_tracker as et
from scipy.signal import savgol_filter
from torchvision import models, transforms
from transformers import CLIPProcessor, CLIPModel
from torchvision.models.segmentation import DeepLabV3_ResNet101_Weights
from transformers import Mask2FormerImageProcessor, Mask2FormerForUniversalSegmentation

Current city is copenhagen!
If you wish to change the city, please edit the value in the __init__.py file


/home/s243636/miniforge3/envs/tese/lib/python3.9/site-packages/heartpy/datautils.py:6: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_filename


Run these cells sequentially

In [3]:
data_root = "/mnt/raid/emotional_data_raquel/data"
fix_root = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/fixations"

_________

# Align gaze data continously with video frames

In [14]:
def align_gaze_continuous(session_path, gaze_csv_path):

    # -----------------------------
    # LOAD DATASET (frames only)
    # -----------------------------
    datapicker = create_datapicker(path=session_path, schema=build_schema) # locate session data
    dataset = load_dataset(datapicker.selected_path, schema=build_schema) # load full dataset structure

    # -----------------------------
    # GAZE (FROM CSV)
    # -----------------------------
    gaze = pd.read_csv(gaze_csv_path, parse_dates=["Seconds"])  # load gaze file

    gaze = gaze.rename(columns={
        "Seconds": "timestamp",
        "GazeX": "gaze_x",
        "GazeY": "gaze_y"
    })

    gaze = gaze[["timestamp", "gaze_x", "gaze_y"]]  # keep only needed columns

    # handle duplicate timestamps
    gaze = gaze.groupby("timestamp", as_index=False).mean() # average gaze if same timestamp appears multiple times

    # set timestamp as index for alignment
    gaze = gaze.set_index("timestamp")

    # remove invalid gaze
    gaze = gaze[(gaze["gaze_x"] >= 0) & (gaze["gaze_y"] >= 0)]

    # ensure sorted (required for merge_asof)
    gaze = gaze.sort_index()

    # -----------------------------
    # FRAMES
    # -----------------------------
    decoded_frames = dataset.streams.PupilLabs.DecodedFrames.data # load frame timestamps
    decoded_frames = decoded_frames[decoded_frames.Value > 0]  # keep only valid frames

    decoded_frames = decoded_frames.copy()  # avoid modifying original data
    decoded_frames["frame"] = np.arange(len(decoded_frames))

    decoded_frames = decoded_frames.sort_index()

    # -----------------------------
    # ALIGN (KEY STEP)
    # -----------------------------
    aligned = pd.merge_asof(
        decoded_frames,              # reference = video frames
        gaze,                        # data to align = gaze
        left_index=True,             # match on frame timestamps
        right_index=True,            # match on gaze timestamps
        direction="nearest",         # take closest gaze sample in time
        tolerance=pd.Timedelta("50ms")  # only match if within 50ms
    )

    # flag frames with valid gaze
    aligned["has_gaze"] = ~aligned["gaze_x"].isna()

    # if drop is necessary
    # aligned = aligned.dropna(subset=["gaze_x", "gaze_y"])

    # -----------------------------
    # FINAL FORMAT
    # -----------------------------
    aligned = aligned.reset_index().rename(columns={"index": "timestamp"})

    return aligned

In [16]:
root = "/mnt/raid/emotional_data_raquel/data"
gaze_root = "/mnt/raid/emotional_data_raquel/supp_mine/gaze"
output_root = "/mnt/raid/emotional_data_raquel/fulldata_analysis_mine/aligned_gaze"

for participant in os.listdir(root):

    participant_path = os.path.join(root, participant)
    participant_label = f"sub-{participant}"

    if not os.path.isdir(participant_path):
        continue

    for session in os.listdir(participant_path):

        session_path = os.path.join(participant_path, session)
        session_name = session.split("_")[1]

        gaze_csv = os.path.join(
            gaze_root,
            participant_label,
            f"ses-{session_name}",
            "gaze.csv"
        )

        if not os.path.exists(gaze_csv):
            print(f"No gaze file for {participant_label}/{session_name}")
            continue

        print(f"Processing {participant_label}/{session_name}")

        try:
            aligned = align_gaze_continuous(session_path, gaze_csv)

            save_path = os.path.join(
                output_root,
                participant,
                session_name,
                "aligned_gaze.csv"
            )

            os.makedirs(os.path.dirname(save_path), exist_ok=True)
            aligned.to_csv(save_path, index=False)

            print(f"Saved → {save_path}")

        except Exception as e:
            print(f"Error: {e}")

Processing sub-OE011/Nordhavn
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/aligned_gaze/OE011/Nordhavn/aligned_gaze.csv
Processing sub-OE015/Norreport
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/aligned_gaze/OE015/Norreport/aligned_gaze.csv
Processing sub-OE019/Hellerup
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/aligned_gaze/OE019/Hellerup/aligned_gaze.csv
Processing sub-OE010/Nordhavn
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/aligned_gaze/OE010/Nordhavn/aligned_gaze.csv
Processing sub-OE024/Nordhavn
Attempting to automatically correct eeg timestamps to harp timestamps...
Done.
Saved → /mnt/raid/emotional_data_raquel/fulldata_analysis_mine/aligned_gaze/OE024/Nordhavn/aligned_g